In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
import joblib

In [20]:
import pandas as pd

# Cargar CSV
df = pd.read_csv("dataset.csv")

# Eliminar columna índice extra si existe
if "Unnamed: 0" in df.columns:
    df = df.drop("Unnamed: 0", axis=1)

# Guardar de nuevo limpio
df.to_csv("dataset_clean.csv", index=False)
print("✅ CSV limpio guardado como dataset_clean.csv")


✅ CSV limpio guardado como dataset_clean.csv


In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
import joblib

# === Cargar dataset limpio ===
df = pd.read_csv("dataset_clean.csv")

# Separar features y etiquetas
X = df.drop(columns=["label", "num_hands"]).values.astype("float32")
y = df["label"].values

# Codificar etiquetas
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_categorical = to_categorical(y_encoded)

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y_categorical, test_size=0.2, random_state=42, stratify=y_categorical
)

# === Definir modelo ===
model = Sequential([
    Dense(256, activation="relu", input_shape=(X_train.shape[1],)),
    Dropout(0.4),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(len(label_encoder.classes_), activation="softmax")
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

# === Entrenamiento ===
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=32,
    verbose=1
)

# === Evaluar ===
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"✅ Precisión en test: {acc*100:.2f}%")

# === Guardar modelo y encoder ===
model.save("sign_model.h5")
joblib.dump(label_encoder, "label_encoder.pkl")
print("✅ Modelo y encoder guardados!")

# === Exportar a TFLite ===
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open("sign_model.tflite", "wb") as f:
    f.write(tflite_model)
print("✅ Exportado a sign_model.tflite")


Epoch 1/30


C:\Users\ledys\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.4292 - loss: 1.6561 - val_accuracy: 1.0000 - val_loss: 0.4518
Epoch 2/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8922 - loss: 0.4556 - val_accuracy: 1.0000 - val_loss: 0.0655
Epoch 3/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9842 - loss: 0.1223 - val_accuracy: 1.0000 - val_loss: 0.0200
Epoch 4/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9921 - loss: 0.0616 - val_accuracy: 1.0000 - val_loss: 0.0068
Epoch 5/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9927 - loss: 0.0379 - val_accuracy: 1.0000 - val_loss: 0.0040
Epoch 6/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9967 - loss: 0.0274 - val_accuracy: 1.0000 - val_loss: 0.0022
Epoch 7/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9991 - loss: 0.0196 - val_accuracy: 1.0000 - val_loss: 0.0014
Epoch 8/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9989 - loss: 0.0122 - val_accuracy: 1.0000 - val_loss: 0.0011
Ep

✅ Precisión en test: 100.00%
✅ Modelo y encoder guardados!
INFO:tensorflow:Assets written to: C:\Users\ledys\AppData\Local\Temp\tmpdvgaohqf\assets


INFO:tensorflow:Assets written to: C:\Users\ledys\AppData\Local\Temp\tmpdvgaohqf\assets


Saved artifact at 'C:\Users\ledys\AppData\Local\Temp\tmpdvgaohqf'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 126), dtype=tf.float32, name='keras_tensor_18')
Output Type:
  TensorSpec(shape=(None, 9), dtype=tf.float32, name=None)
Captures:
  2327556939744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2327556944848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2327556938688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2327556944144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2327556943440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2327556947488: TensorSpec(shape=(), dtype=tf.resource, name=None)
✅ Exportado a sign_model.tflite


In [15]:
print("Input details:", input_details)
print("Output details:", output_details)


Input details: [{'name': 'serving_default_keras_tensor_6:0', 'index': 0, 'shape': array([  1, 127]), 'shape_signature': array([ -1, 127]), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
Output details: [{'name': 'StatefulPartitionedCall_1:0', 'index': 10, 'shape': array([1, 5]), 'shape_signature': array([-1,  5]), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]


In [11]:
print("Shape de dataset limpio:", df.shape)
print("Columnas:", df.columns.tolist())


Shape de dataset limpio: (1400, 128)
Columnas: ['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11', 'f12', 'f13', 'f14', 'f15', 'f16', 'f17', 'f18', 'f19', 'f20', 'f21', 'f22', 'f23', 'f24', 'f25', 'f26', 'f27', 'f28', 'f29', 'f30', 'f31', 'f32', 'f33', 'f34', 'f35', 'f36', 'f37', 'f38', 'f39', 'f40', 'f41', 'f42', 'f43', 'f44', 'f45', 'f46', 'f47', 'f48', 'f49', 'f50', 'f51', 'f52', 'f53', 'f54', 'f55', 'f56', 'f57', 'f58', 'f59', 'f60', 'f61', 'f62', 'f63', 'f64', 'f65', 'f66', 'f67', 'f68', 'f69', 'f70', 'f71', 'f72', 'f73', 'f74', 'f75', 'f76', 'f77', 'f78', 'f79', 'f80', 'f81', 'f82', 'f83', 'f84', 'f85', 'f86', 'f87', 'f88', 'f89', 'f90', 'f91', 'f92', 'f93', 'f94', 'f95', 'f96', 'f97', 'f98', 'f99', 'f100', 'f101', 'f102', 'f103', 'f104', 'f105', 'f106', 'f107', 'f108', 'f109', 'f110', 'f111', 'f112', 'f113', 'f114', 'f115', 'f116', 'f117', 'f118', 'f119', 'f120', 'f121', 'f122', 'f123', 'f124', 'f125', 'num_hands', 'label']


In [12]:
print("Shape de X:", X.shape)


Shape de X: (1400, 127)


#### Probar Aleatorio

In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow as tf

# === CARGAR MODELO Y DATASET ===
model = tf.keras.models.load_model("sign_model.h5")
data = pd.read_csv("dataset.csv")

# Preparar datos
X = data.drop("label", axis=1).values
y = data["label"].values

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# === PROBAR UNA PREDICCIÓN ===
# Toma una muestra al azar
idx = np.random.randint(0, len(X_scaled))
sample = X_scaled[idx].reshape(1, -1)

# Predicción
pred_probs = model.predict(sample)
pred_class = np.argmax(pred_probs)

print(f"Muestra real: {y[idx]}")
print(f"Predicción: {encoder.inverse_transform([pred_class])[0]}")
print(f"Probabilidades: {pred_probs}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
Muestra real: T
Predicción: T
Probabilidades: [[2.0844550e-07 5.7014995e-07 2.4701472e-09 2.0658383e-13 1.5210299e-09
  3.7169612e-12 1.6453714e-06 1.9995772e-09 1.2083322e-13 3.7026886e-16
  2.8022990e-11 3.1882461e-10 5.1027456e-07 3.0299267e-11 8.2856822e-08
  9.9169173e-10 9.9999678e-01 1.4040929e-13 2.6751490e-16 1.2742642e-12
  1.9647517e-08 9.3752369e-08]]


#### Probar Camera

In [23]:
import pandas as pd
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
import joblib

# === Construir mapa de requerimiento de manos desde dataset ===
df = pd.read_csv("dataset_clean.csv")
label_hand_requirement = {}

for label in df["label"].unique():
    subset = df[df["label"] == label].drop("label", axis=1).values
    # Ver si en alguna muestra la segunda mano tiene valores != 0
    second_hand = subset[:, 63:126]  # segunda mano
    if np.any(second_hand != 0):
        label_hand_requirement[label] = 2
    else:
        label_hand_requirement[label] = 1

print("Mapa de señas:", label_hand_requirement)

# === Cargar modelo TFLite y encoder ===
interpreter = tf.lite.Interpreter(model_path="sign_model.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

label_encoder = joblib.load("label_encoder.pkl")

# === Configuración MediaPipe ===
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

# Función: normalizar landmarks
def normalize_landmarks(hand_landmarks):
    coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark])
    wrist = coords[0]
    coords -= wrist
    scale = np.linalg.norm(coords[9])
    if scale > 0:
        coords /= scale
    return coords.flatten().tolist()

# === Captura de cámara ===
cap = cv2.VideoCapture(0)

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
) as hands:

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(frame_rgb)
        frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

        row = []
        num_hands = 0
        if results.multi_hand_landmarks:
            num_hands = len(results.multi_hand_landmarks)
            for hand_landmarks in results.multi_hand_landmarks:
                row.extend(normalize_landmarks(hand_landmarks))
                mp_drawing.draw_landmarks(frame_bgr, hand_landmarks, mp_hands.HAND_CONNECTIONS)

        while len(row) < 126:
            row.append(0.0)

        if any(val != 0.0 for val in row):
            input_data = np.array([row], dtype=np.float32)
            interpreter.set_tensor(input_details[0]['index'], input_data)
            interpreter.invoke()
            output_data = interpreter.get_tensor(output_details[0]['index'])

            pred_index = np.argmax(output_data)
            pred_label = label_encoder.inverse_transform([pred_index])[0]
            confidence = np.max(output_data)

            # === Validación estricta ===
            required_hands = label_hand_requirement.get(pred_label, 1)
            if num_hands == required_hands:
                cv2.putText(frame_bgr, f"{pred_label} ({confidence:.2f})",
                            (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
            else:
                cv2.putText(frame_bgr, "N/A", (30, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

        cv2.imshow("Test Sign Model", frame_bgr)

        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()


Mapa de señas: {'A': 1, 'B': 1, 'C': 1, 'D': 1, 'Jesus': 2, 'Dios': 1, 'Diablo': 1, 'Y': 1, 'Alma': 2}


C:\Users\ledys\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


ValueError: Cannot set tensor: Dimension mismatch. Got 189 but expected 126 for dimension 1 of input 0.